<a href="https://colab.research.google.com/github/adityab-tech/Prism/blob/main/W5_Prism.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import os
import glob
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import torch
import random
import yaml #yeh ek lib ki tarah hai jo saara data store karti hai .. pyyaml se yaml file load hoti hai
from torch.utils.data import Dataset, DataLoader

In [2]:
!pip install -q transformers accelerate sentencepiece safetensors pyyaml peft huggingface_hub

In [3]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [4]:
!git clone https://github.com/shiyu-coder/Kronos.git

fatal: destination path 'Kronos' already exists and is not an empty directory.


In [5]:
%cd /content/Kronos

/content/Kronos


In [6]:
!ls

config.yaml  figures	   Kronos   README.md	      webui
data	     finetune	   LICENSE  requirements.txt
examples     finetune_csv  model    tests


In [7]:
df = pd.read_csv(
"/content/drive/MyDrive/PRISM/processed/RELIANCE.NS_processed.csv")
df.head()

,Date,Close,High,Low,Open,Volume,return,log_return,MA5,MA10,MA20,Volatility,Volume_Change,Volume_MA20,market_return,beta,target
0,2019-04-02,615.133118,621.064466,610.883802,618.807021,17526740,-0.001545,-0.001546,607.125757,603.150867,587.337653,0.016157,-0.206535,21556498.85,0.003775,1.569316,-0.010434
1,2019-04-03,608.714844,621.020165,607.298386,616.483114,17169813,-0.010434,-0.010489,607.829541,604.264093,590.638626,0.016430,-0.020365,21548509.20,-0.005912,1.593855,-0.016107
2,2019-04-04,598.910339,612.477168,596.342998,610.396773,18320845,-0.016107,-0.016238,608.165942,603.223895,593.192639,0.017118,0.067038,21685676.45,-0.003946,1.623214,0.000628
3,2019-04-05,599.286560,603.712925,594.461833,602.407152,14717266,0.000628,0.000628,607.625916,602.270007,595.164584,0.016639,-0.196693,21104925.95,0.005859,1.608380,-0.018207
4,2019-04-08,588.375610,600.880142,585.919014,600.216205,19081843,-0.018207,-0.018374,602.084094,601.716705,596.470364,0.017331,0.296562,21172113.50,-0.005267,1.638698,0.003912


In [8]:
df = df.rename(columns={
    "Date":"timestamps",
    "Open":"open",
    "High":"high",
    "Low":"low",
    "Close":"close",
    "Volume":"volume"
})
df["amount"] = 0         #kyuki OHLC jaruri hai Vol and Amt can be 0

In [9]:
df["timestamps"] = pd.to_datetime(df["timestamps"])
df.tail()

,timestamps,close,high,low,open,volume,return,log_return,MA5,MA10,MA20,Volatility,Volume_Change,Volume_MA20,market_return,beta,target,amount
1412,2024-12-23,1211.834717,1216.692666,1202.812534,1204.597171,10052824,0.014104,0.014006,1220.777417,1241.647192,1266.351343,0.012267,-0.505101,14831684.80,0.007035,1.320625,0.000368,0
1413,2024-12-24,1212.280762,1222.988340,1210.545745,1211.834663,6734917,0.000368,0.000368,1216.306030,1235.490369,1262.735071,0.012095,-0.330047,14706052.80,-0.001086,1.337080,-0.005070,0
1414,2024-12-26,1206.133911,1217.188348,1203.853555,1213.767935,10016178,-0.005070,-0.005083,1209.028882,1229.378162,1258.935388,0.012103,0.487201,14728224.40,0.000950,1.332965,0.003699,0
1415,2024-12-27,1210.595459,1217.386785,1206.580087,1207.869004,7000397,0.003699,0.003692,1207.165015,1225.229004,1256.469189,0.011698,-0.301091,14326873.65,0.002661,1.295769,-0.008476,0
1416,2024-12-30,1200.333984,1212.726960,1197.756270,1205.985254,8818766,-0.008476,-0.008513,1208.235767,1219.067212,1252.429083,0.010903,0.259752,14107696.60,-0.007076,1.294135,0.003923,0


In [10]:
save_path="/content/Kronos/data"
os.makedirs(save_path,exist_ok=True)
df.to_csv(f"{save_path}/reliance.csv",
index=False)
#df ko /content/Kronos/data/reliance.csv naam ki CSV file me save kar do and csv ke indexes save mat karna. Agar data folder nahi hai to pehle usse bana do.

In [11]:
pd.read_csv(
"/content/Kronos/data/reliance.csv").head()

,timestamps,close,high,low,open,volume,return,log_return,MA5,MA10,MA20,Volatility,Volume_Change,Volume_MA20,market_return,beta,target,amount
0,2019-04-02,615.133118,621.064466,610.883802,618.807021,17526740,-0.001545,-0.001546,607.125757,603.150867,587.337653,0.016157,-0.206535,21556498.85,0.003775,1.569316,-0.010434,0
1,2019-04-03,608.714844,621.020165,607.298386,616.483114,17169813,-0.010434,-0.010489,607.829541,604.264093,590.638626,0.016430,-0.020365,21548509.20,-0.005912,1.593855,-0.016107,0
2,2019-04-04,598.910339,612.477168,596.342998,610.396773,18320845,-0.016107,-0.016238,608.165942,603.223895,593.192639,0.017118,0.067038,21685676.45,-0.003946,1.623214,0.000628,0
3,2019-04-05,599.286560,603.712925,594.461833,602.407152,14717266,0.000628,0.000628,607.625916,602.270007,595.164584,0.016639,-0.196693,21104925.95,0.005859,1.608380,-0.018207,0
4,2019-04-08,588.375610,600.880142,585.919014,600.216205,19081843,-0.018207,-0.018374,602.084094,601.716705,596.470364,0.017331,0.296562,21172113.50,-0.005267,1.638698,0.003912,0


In [12]:
#puri settings ka control pannel
config={
"data":{                                                 #dataset se related settings
"data_path":"/content/Kronos/data/reliance.csv",
"lookback_window":30,                 #30 pechle data tak dekh sakta hai
"predict_window":5,                   #predict next 5
"max_context":30
},

"training":{                                               #training se related settings.
"batch_size":32,
"epochs":5,
"learning_rate":1e-4
},

"model_paths":{                                            #model aur tokenizer ke paths.
"pretrained_tokenizer":"NeoQuasar/Kronos-Tokenizer-base",
"pretrained_predictor":"NeoQuasar/Kronos-base",
"base_save_path":"/content/Kronos/checkpoints",            #trained model yahan save hoga
"exp_name":"Reliance"
}
}

In [13]:
#YAML is used to separate configuration from code.
# It stores hyperparameters, dataset paths, and model settings in one place, making experiments easier to modify, reproduce, and manage without changing the training code.

In [14]:
#config dictionary ko config.yaml naam ki YAML file me save kar do
with open("/content/Kronos/config.yaml","w") as f:
    yaml.dump(config,f)

In [15]:
os.path.exists(
"/content/Kronos/config.yaml")

True

In [16]:
from finetune_csv.config_loader import CustomFinetuneConfig
config=CustomFinetuneConfig("/content/Kronos/config.yaml")
#Ye class YAML config file ko read karke Python object banati hai.

In [17]:
config.print_config_summary()
#print_config_summary()→uss class ka ek function hai

Kronos finetuning configuration summary
Experiment name: Reliance
Data path: /content/Kronos/data/reliance.csv
Lookback window: 30
Predict window: 5
Tokenizer training epochs: 5
Basemodel training epochs: 5
Batch size: 32
Tokenizer learning rate: 0.0002
Predictor learning rate: 4e-05
Train tokenizer: True
Train basemodel: True
Skip existing: False
Use pre-trained tokenizer: True
Use pre-trained predictor: True
Base save path: /content/Kronos/checkpoints
Tokenizer save path: /content/Kronos/checkpoints/tokenizer
Basemodel save path: /content/Kronos/checkpoints/basemodel


In [18]:
import sys
sys.path.append("/content/Kronos")
sys.path.append("/content/Kronos/finetune_csv")
#saanp(py) ko bata do ki Kronos aur finetune_csv folders me bhi modules dhund na
#sys.path.append() = Python ke module search path me naya folder add karo taaki us folder ki .py files import ho sake

In [19]:
#CSV data ko train/validation dataLoaders aur datasets me convert karne wala fn
from finetune_csv.finetune_base_model import create_dataloaders

In [20]:
#create_dataloaders(config) =dataset banao → train/val mai split karo →samplers banao→ dataLoaders banao →sab return kar do
train_loader,val_loader,train_dataset,val_dataset,train_sampler,val_sampler=\
create_dataloaders(config)
#sampler--decide karta hai data kis order mai lena hai.

Creating data loaders...
Original data time range: 2019-04-02 00:00:00 to 2024-12-30 00:00:00
Original data total length: 1417 records
[TRAIN] Training set: first 1275 time points (0.9)
[TRAIN] Training set time range: 2019-04-02 00:00:00 to 2024-06-04 00:00:00
[TRAIN] Data length after split: 1275 records
[TRAIN] Data length: 1275, Available samples: 1240
Original data time range: 2019-04-02 00:00:00 to 2024-12-30 00:00:00
Original data total length: 1417 records
[VAL] Validation set: time points 1276 to 1417 (0.1)
[VAL] Validation set time range: 2024-06-05 00:00:00 to 2024-12-30 00:00:00
[VAL] Data length after split: 142 records
[VAL] Data length: 142, Available samples: 107
Training set size: 1240, Validation set size: 107


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:424: UserWarning: This DataLoader will create 6 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()


In [21]:
print(len(train_dataset))
print(len(val_dataset))

1240
107


In [22]:
x,x_stamp=train_dataset[0]
#x = stock data while x_stamp = uss data ke time features

In [23]:
print(x.shape)
print(x_stamp.shape)

torch.Size([36, 6])
torch.Size([36, 5])


In [24]:
print(x[:5])
print(x_stamp[:4])

tensor([[-0.7396, -0.7876, -0.8774, -0.9850, -0.2669,  0.0000],
        [-0.9897, -1.0763, -1.1095, -1.1571, -0.5279,  0.0000],
        [-1.1404, -1.2752, -1.3149, -1.2961, -0.1821,  0.0000],
        [-1.1725, -1.1483, -1.0983, -1.1709, -0.0373,  0.0000],
        [-1.1404, -1.1982, -1.1964, -1.3123, -0.4402,  0.0000]])
tensor([[ 0.,  0.,  0., 26.,  7.],
        [ 0.,  0.,  1., 27.,  7.],
        [ 0.,  0.,  2., 28.,  7.],
        [ 0.,  0.,  3., 29.,  7.]])


In [25]:
!pwd
!ls /content

/content/Kronos
drive  Kronos  sample_data


In [26]:
import model
print(model.__file__)

/content/Kronos/model/__init__.py


In [27]:
!cat /content/Kronos/model/__init__.py        #__init__.py → Package ke important classes export karta ha

from .kronos import KronosTokenizer, Kronos, KronosPredictor

model_dict = {
    'kronos_tokenizer': KronosTokenizer,
    'kronos': Kronos,
    'kronos_predictor': KronosPredictor
}


def get_model_class(model_name):
    if model_name in model_dict:
        return model_dict[model_name]
    else:
        print(f"Model {model_name} not found in model_dict")
        raise NotImplementedError




In [28]:
from model import Kronos,KronosTokenizer

In [29]:
tokenizer=KronosTokenizer.from_pretrained("NeoQuasar/Kronos-Tokenizer-base")

In [30]:
model=Kronos.from_pretrained("NeoQuasar/Kronos-base")

In [31]:
print(model) #yeh arch batayega

Kronos(
  (token_drop): Dropout(p=0.0, inplace=False)
  (embedding): HierarchicalEmbedding(
    (emb_s1): Embedding(1024, 832)
    (emb_s2): Embedding(1024, 832)
    (fusion_proj): Linear(in_features=1664, out_features=832, bias=True)
  )
  (time_emb): TemporalEmbedding(
    (minute_embed): Embedding(60, 832)
    (hour_embed): Embedding(24, 832)
    (weekday_embed): Embedding(7, 832)
    (day_embed): Embedding(32, 832)
    (month_embed): Embedding(13, 832)
  )
  (transformer): ModuleList(
    (0-11): 12 x TransformerBlock(
      (norm1): RMSNorm()
      (self_attn): MultiHeadAttentionWithRoPE(
        (q_proj): Linear(in_features=832, out_features=832, bias=True)
        (k_proj): Linear(in_features=832, out_features=832, bias=True)
        (v_proj): Linear(in_features=832, out_features=832, bias=True)
        (out_proj): Linear(in_features=832, out_features=832, bias=True)
        (rotary): RotaryPositionalEmbedding()
        (resid_dropout): Dropout(p=0.2, inplace=False)
      )
    

In [32]:
for param in model.parameters():
    param.requires_grad=False

In [33]:
for param in model.head.parameters():
    param.requires_grad=True

In [34]:
sum(p.requires_grad for p in model.parameters())

4

In [35]:
optimizer=torch.optim.AdamW(
filter(lambda p:p.requires_grad,
model.parameters()),
lr=1e-4)

In [36]:
criterion=torch.nn.MSELoss()

In [37]:
next(iter(train_loader))[0].shape

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:432: UserWarning: This DataLoader will create 6 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:1118: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


torch.Size([32, 36, 6])

In [38]:
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total Parameters     : {total_params:,}")
print(f"Trainable Parameters : {trainable_params:,}")

Total Parameters     : 102,310,592
Trainable Parameters : 1,705,984


In [39]:
x, x_stamp = next(iter(train_loader))
print(x.shape)
print(x_stamp.shape)

torch.Size([32, 36, 6])
torch.Size([32, 36, 5])


In [40]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
x = x.to(device)
x_stamp = x_stamp.to(device)
model = model.to(device)
print("Device :", device)

Device : cpu


In [41]:
print("Train Dataset Size :", len(train_dataset))
print("Validation Dataset Size :", len(val_dataset))
x, x_stamp = train_dataset[0]

print("Input Shape :", x.shape)
print("Time Shape  :", x_stamp.shape)

Train Dataset Size : 1240
Validation Dataset Size : 107
Input Shape : torch.Size([36, 6])
Time Shape  : torch.Size([36, 5])


In [42]:
print("Input dtype :", x.dtype)
print("Time dtype  :", x_stamp.dtype)
print()
print("Min :", x.min())
print("Max :", x.max())

Input dtype : torch.float32
Time dtype  : torch.float32

Min : tensor(-1.3149)
Max : tensor(3.2781)


In [43]:
batch_x, batch_stamp = next(iter(train_loader))
batch_x = batch_x.to(device)
batch_stamp = batch_stamp.to(device)

print(batch_x.shape)
print(batch_stamp.shape)

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:432: UserWarning: This DataLoader will create 6 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:1118: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


torch.Size([32, 36, 6])
torch.Size([32, 36, 5])


In [44]:
print("Batch successfully loaded.")
print("Ready for Kronos official training pipeline.")

Batch successfully loaded.
Ready for Kronos official training pipeline.
